In [3]:
# =========================
# IMPORTS
# =========================
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

# =========================
# LOAD DATA
# =========================
kmnist = fetch_openml('Kuzushiji-MNIST', version=1)

X = kmnist.data.astype('float32') / 255.0
y = kmnist.target.astype('int')

X = X.values.reshape(-1, 28, 28, 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=10000, random_state=42
)

# =========================
# MODEL FUNCTION
# =========================
def create_model(lr=0.001, dropout_rate=0.3, filters=32):
    model = keras.Sequential([
        keras.Input(shape=(28,28,1)),

        layers.Conv2D(filters, (3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),

        layers.Conv2D(filters*2, (3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),

        layers.Flatten(),

        layers.Dense(128, activation='relu'),
        layers.Dropout(dropout_rate),

        layers.Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# =========================
# EXPERIMENTS
# =========================
configs = [
    {"lr":0.01, "batch":32, "dropout":0.2, "filters":32},
    {"lr":0.001, "batch":64, "dropout":0.3, "filters":32},
    {"lr":0.001, "batch":64, "dropout":0.5, "filters":64},
    {"lr":0.0005, "batch":128, "dropout":0.3, "filters":64},
]

results = []

for i, cfg in enumerate(configs):
    print(f"\nRunning Experiment {i+1}: {cfg}")

    model = create_model(
        lr=cfg["lr"],
        dropout_rate=cfg["dropout"],
        filters=cfg["filters"]
    )

    model.fit(
        X_train, y_train,
        epochs=5,
        batch_size=cfg["batch"],
        verbose=0
    )

    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    print("Accuracy:", acc)

    results.append((cfg, acc))

# =========================
# BEST RESULT
# =========================
best = max(results, key=lambda x: x[1])

print("\n=== BEST CONFIGURATION ===")
print("Config:", best[0])
print("Accuracy:", best[1])

2026-04-14 14:03:51.978856: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776175432.322200      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776175432.416516      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776175433.202575      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776175433.202636      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776175433.202640      55 computation_placer.cc:177] computation placer alr


Running Experiment 1: {'lr': 0.01, 'batch': 32, 'dropout': 0.2, 'filters': 32}


2026-04-14 14:04:43.596400: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Accuracy: 0.9567000269889832

Running Experiment 2: {'lr': 0.001, 'batch': 64, 'dropout': 0.3, 'filters': 32}
Accuracy: 0.9817000031471252

Running Experiment 3: {'lr': 0.001, 'batch': 64, 'dropout': 0.5, 'filters': 64}
Accuracy: 0.9836000204086304

Running Experiment 4: {'lr': 0.0005, 'batch': 128, 'dropout': 0.3, 'filters': 64}
Accuracy: 0.9793999791145325

=== BEST CONFIGURATION ===
Config: {'lr': 0.001, 'batch': 64, 'dropout': 0.5, 'filters': 64}
Accuracy: 0.9836000204086304


Experiment 1:

With a high learning rate of 0.01, batch size 32, and dropout 0.2, the model achieved 95.67% accuracy.
Compared to the best model (98.36%), this is ~2.69% lower, indicating unstable learning due to a high learning rate.

Experiment 2:

Reducing the learning rate to 0.001 and increasing batch size to 64 improved accuracy to 98.17%, which is +2.50% higher than Experiment 1.
This shows that a lower learning rate leads to more stable convergence.

Experiment 3 (BEST):

Increasing filters to 64 and dropout to 0.5 resulted in the highest accuracy of 98.36%, which is:

+0.19% higher than Experiment 2
+2.69% higher than Experiment 1

This indicates that:

More filters improve feature extraction
Higher dropout reduces overfitting
Experiment 4:

Reducing learning rate further to 0.0005 and increasing batch size to 128 resulted in 97.94% accuracy, which is ~0.42% lower than the best model.
This suggests that too low learning rates slow down learning.

Learning Rate:

Reducing learning rate from 0.01 → 0.001 improved accuracy by +2.50%, showing better convergence.

Filters:

Increasing filters from 32 → 64 improved accuracy by ~0.19%, showing better feature extraction.

Dropout:

Increasing dropout from 0.3 → 0.5 slightly improved accuracy by ~0.19%, indicating reduced overfitting.

Batch Size:

Increasing batch size from 32 → 64 improved performance, but further increase to 128 slightly reduced accuracy.

The best configuration was achieved with a learning rate of 0.001, batch size of 64, filter size of 64, and dropout rate of 0.5, achieving an accuracy of 98.36%. This configuration provides a balance between stable learning, effective feature extraction, and overfitting control. A higher learning rate led to unstable training, while very low learning rates slowed convergence.

The experiments were computationally intensive and required significant execution time, highlighting the trade-off between model performance and computational cost.